# Basic Settings

In [5]:
### 한글 폰트 설치
!apt-get install -y fonts-nanum
!fc-cache -fv
!rm ~/.cache/matplotlib -rf
# 설치 후 colab의 경우 Runtime > Restart session 필요

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
Font directories:
	/root/.local/share/fonts
	/usr/local/share/fonts
	/usr/share/fonts
	/root/.fonts
	/usr/share/fonts/truetype
	/usr/share/fonts/truetype/dejavu
	/usr/share/fonts/truetype/nanum
/root/.local/share/fonts: skipping, no such directory
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 2 dirs
/usr/share/fonts/truetype/dejavu: caching, new cache contents: 22 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/dejavu: skipping, looped directory detecte

In [6]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('font', family='NanumBarunGothic') # 혹은 다른 설치한 Nanum 폰트 사용

In [7]:
import pandas as pd
import seaborn as sns
import numpy as np
import scipy
import scipy.stats as stats

# 분석 : 전국 합계출산율과 출생아 수 현황 분석

## 분석 목적 : 전국 총합의 합계출산율과 출생아 수 현황을 분석하여 전국적으로 출산율 감소가 어느정도 심각성을 보이는지 확인

### dataset 불러오기

In [25]:
# 전국 data 불러오기

# CSV 파일 경로 지정
file_path01 = '../../dataset/National data/National births.csv'
file_path02 = '../../dataset/National data/Total national fertility rate.csv'

# CSV 파일을 DataFrame으로 읽어오기
df_national_births = pd.read_csv(file_path01, encoding='utf-8')
df_national_fertility_rate = pd.read_csv(file_path02, encoding='utf-8')

In [26]:
df_national_births.head()

,시군구별,2021.05,2021.06,2021.07,2021.08,2021.09,2021.10,2021.11,2021.12,2022.01,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,시군구별,계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),...,계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명),계 (명)
1,전국,21922,21504,22364,22282,21905,20749,19829,17179,24637,...,436455,435435,438420,406243,357771,326822,302676,272337,260562,249186


In [27]:
df_national_fertility_rate.head()

,시군구별,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,전국,1.297,1.187,1.205,1.239,1.172,1.052,0.977,0.918,0.837,0.808,0.778


### data 전처리

In [28]:
# 출생아 수 data 0번째 행 삭제
df_national_births = df_national_births.drop(0).reset_index(drop=True)
df_national_births

,시군구별,2021.05,2021.06,2021.07,2021.08,2021.09,2021.10,2021.11,2021.12,2022.01,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,전국,21922,21504,22364,22282,21905,20749,19829,17179,24637,...,436455,435435,438420,406243,357771,326822,302676,272337,260562,249186


In [29]:
# 열 이름을 파싱하여 연도와 월로 분리
def parse_date(column_name):
    try:
        year, month = column_name.split('.')
        return int(year), int(month)
    except ValueError:
        return None

In [30]:
# 열 이름 변경
df_national_births.columns = [parse_date(col) if parse_date(col) is not None else col for col in df_national_births.columns]

# '시군구별' 열을 제외한 나머지 열은 날짜와 관련이 있습니다
# '시군구별' 열을 제외한 데이터 부분
df_data = df_national_births.drop(columns=['시군구별'])

# 연도별 합계출산율을 계산하기 위해 연도를 기준으로 그룹화
# 먼저 열을 '연도'와 '월'로 나눠서 연도별로 집계할 수 있도록 전처리
df_data.columns = pd.MultiIndex.from_tuples(df_data.columns)

# 열을 연도별로 그룹화하여 합계 계산
df_yearly_sum = df_data.groupby(level=0, axis=1).sum()

# 결과 확인
print(df_yearly_sum)

TypeError: Expected tuple, got str